# 02_reg_single — fit (단일 회귀모델, MODEL_NAME 스위치)

`02_reg_single/hpo.py --model {MODEL_NAME}` 병렬 HPO가 만든 Optuna study에서 best trial을 로드해
K-fold refit → die/unit 산출물 번들을 저장한다. `MODEL_NAME`만 바꾸면 5종(lgbm/xgb/catboost/et/enet) 모두 처리.

- 전처리: `zit_pp.load_for_fit` (cleaning.py 정본, 전 트랙 공용 PP).
- refit/저장: `modules.hpo.refit_best` + `save_artifacts` (enet은 fold-local RobustScaler 내장).
- 산출물: §5.1 의미 폴더 `4_output/02_reg_single/{MODEL_NAME}/`.

## 1. 환경 설정

In [1]:
import os, sys
RESUME = True   # 기존 Optuna study(db)에 이어서 학습할지 (필요시 config 셀에서 덮어씀)
# ── Colab이면 코드 번들 1개(code.zip)만 받아 풀기 — 데이터·경로·폰트는 setup.py가 처리 ──
try:
    import google.colab  # Colab에서만 import 성공
    GDRIVE_CODE_ID = '1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I'  # code.zip = setup.py+requirements+utils+2_preprocessing+3_modeling 지원코드
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip -q install gdown')
        os.system(f'gdown {GDRIVE_CODE_ID} -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
    os.chdir('/content/project')
except ImportError:
    pass
# ── 공통: cwd에서 위로 setup.py(+utils/)를 자동탐색해 실행 (노트북 깊이·드라이브 위치 무관) ──
_d = os.getcwd()
while not (os.path.exists(os.path.join(_d, 'setup.py')) and os.path.isdir(os.path.join(_d, 'utils'))):
    _p = os.path.dirname(_d)
    if _p == _d:
        raise RuntimeError('프로젝트 루트(setup.py + utils/)를 못 찾음 — cwd 확인')
    _d = _p
if _d not in sys.path:
    sys.path.insert(0, _d)
import runpy
runpy.run_path(os.path.join(_d, 'setup.py'))

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# 공통 유틸: 경로 상수(OUTPUT_DIR, DATA_DIR), 컬럼 상수(TARGET_COL, KEY_COL), SEED
from utils.config import PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, OUTPUT_DIR
from utils.data import load_all, get_feat_cols, split_xs

# 전처리 모듈(2_preprocessing) 경로 + `from modules import ...` 가 3_modeling/modules를 찾게
# 전처리 모듈(2_preprocessing/)과 모델링 모듈(3_modeling/modules/) 모두 sys.path 등록
# 노트북 위치가 달라도 PROJECT_ROOT 기준 절대경로로 접근 → Colab·로컬 동일
PREP_ROOT = os.path.join(PROJECT_ROOT, '2_preprocessing')
if PREP_ROOT not in sys.path:
    sys.path.insert(0, PREP_ROOT)
MODEL_ROOT = os.path.join(PROJECT_ROOT, '3_modeling')
if MODEL_ROOT not in sys.path:
    sys.path.insert(0, MODEL_ROOT)

# preprocess.run: 전체 전처리 파이프라인(결측/이상치/스케일/집계)
# hpo: HPO(run_hpo) + 재학습(refit_best) + 산출물 저장(save_artifacts)
# models: 모델 레지스트리 (AVAILABLE_MODELS 리스트 + 모델 생성 팩토리)
from modules import preprocess, hpo, models   # noqa: E402  (preprocess.run, hpo.run_hpo/refit_best/save_artifacts, models 레지스트리)
# meta_features: position·die_xy 메타피처 생성 (run_wf_xy 파싱 기반)
from meta_features import add_meta_features

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'Available models: {models.AVAILABLE_MODELS}')

setup 완료


PROJECT_ROOT = C:\Users\Dell5371\Desktop\기업연계프로젝트
Available models: ['lgbm', 'xgb', 'catboost', 'et', 'enet', 'zitboost']


## 2. 실험 설정 — MODEL_NAME 스위치 + 소스 study + 후처리

In [ ]:
# 모델 선택 — 이 fit 노트북은 MODEL_NAME만 바꾸면 5종 모두 처리한다.
MODEL_NAME = 'lgbm'   # {'lgbm','xgb','catboost','et','enet'}
USER = 'jh'

# 소스 study — 02_reg_single/hpo.py --model {MODEL_NAME} 가 만든 것.
SOURCE_STUDY_NAME = f'reg_{MODEL_NAME}'
SOURCE_DB_DIR = os.path.join(OUTPUT_DIR, '02_reg_single', MODEL_NAME)
SOURCE_DB_PATH = os.path.join(SOURCE_DB_DIR, f'optuna_{USER}_{SOURCE_STUDY_NAME}.db')

N_FOLDS = 5
N_JOBS = -1
# 트리·enet 통일: target 변환 없음 (트리는 log1p와 사실상 동등, enet도 'none' 기본).
TARGET_TRANSFORM = 'none'
CLIP_Y_EXTREME = True

# 산출물 위치 (§5.1 의미 폴더, 실험번호 없음). stacking discovery가 이 leaf에서 *_die.csv를 찾는다.
EXP_ID = SOURCE_STUDY_NAME
OUT_DIR = os.path.join(OUTPUT_DIR, '02_reg_single', MODEL_NAME)
os.makedirs(OUT_DIR, exist_ok=True)

# 후처리: die->unit 집계 8종 + zero_clip 그리드 -> val RMSE 최소 조합 (save_artifacts가 적용).
POSTPROCESS_CONFIG = {
    'agg_methods': ('mean', 'median', 'max', 'min', 'trimmed_mean', 'weighted', 'Q25', 'Q75'),
    'zero_clip_range': (0.001, 0.015),
    'zero_clip_step': 0.001,
    'zero_clip_log_space': TARGET_TRANSFORM == 'log1p',
    'use_pi_threshold': False,
}
print(f'MODEL_NAME={MODEL_NAME} | study={SOURCE_STUDY_NAME}')
print(f'SOURCE_DB_PATH={SOURCE_DB_PATH}')
print(f'OUT_DIR={OUT_DIR}')

## 3. best trial 로드 + 데이터 (통일 PP)

In [ ]:
import ast
from pathlib import Path
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)
from modules import hpo  # refit_best / save_artifacts

# zit_pp(공용 PP) import 경로 추가.
_ZIT_DIR = os.path.join(PROJECT_ROOT, '3_modeling', '01_zit')
if _ZIT_DIR not in sys.path:
    sys.path.insert(0, _ZIT_DIR)
import zit_pp


def _parse_user_attr(v):
    if isinstance(v, str):
        try:
            return ast.literal_eval(v)
        except (ValueError, SyntaxError):
            return v
    return v


# 새 study에서 best trial 로드. best_params(raw)는 refit_best가 _hp_from_best로 해석한다(already_resolved=False).
_storage = f'sqlite:///{Path(SOURCE_DB_PATH).as_posix()}'
_study = optuna.load_study(study_name=SOURCE_STUDY_NAME, storage=_storage)
_best = _study.best_trial
_ua = {k: _parse_user_attr(v) for k, v in _study.user_attrs.items()}
best_params = dict(_best.params)
study_meta_for_save = {
    'exp_id': EXP_ID,
    'model_name': MODEL_NAME,
    'best_trial_number': _best.number,
    'best_oof_rmse': float(_best.value),
    'study_meta': dict(_ua),
}
print(f'[best] study={SOURCE_STUDY_NAME} trial#{_best.number} oof={_best.value:.9f}')

# 데이터 — 통일 PP (zit_pp.load_for_fit, cleaning.py 정본).
_d = zit_pp.load_for_fit(clip_y_extreme=CLIP_Y_EXTREME)
xs_train, xs_val, xs_test = _d['xs_train'], _d['xs_val'], _d['xs_test']
ys_input = _d['ys_input']
feat_cols_clean = _d['feat_cols']

# target 변환 (TARGET_TRANSFORM='none' -> identity / None).
target_transform_fn = None
target_inverse_fn = None
print(f'[data] feat={len(feat_cols_clean)}, '
      f"train_units={len(_d['y_train_unit_s']):,}, val={len(_d['y_val_unit_s']):,}, test={len(_d['y_test_unit_s']):,}")

## 4. Best refit (K-fold OOF)

In [ ]:
# best HP로 K-fold 재학습 -> die-level OOF / val / test 예측 (val·test는 fold 평균).
final = hpo.refit_best(
    xs_train=xs_train, xs_val=xs_val, xs_test=xs_test,
    ys_train_unit=ys_input['train'],
    feat_cols=feat_cols_clean,
    model_name=MODEL_NAME,
    best_params=best_params,
    n_folds=N_FOLDS,
    n_jobs=N_JOBS,
    target_transform_fn=target_transform_fn,
    target_inverse_fn=target_inverse_fn,
)

# 후처리 이전(mean 집계) unit RMSE — train(OOF) / val / test.
y_true = ys_input['train'].set_index(KEY_COL)[TARGET_COL]
oof_u = final['oof_pred_unit'].set_index(KEY_COL)['pred'].loc[y_true.index]
oof_rmse = float(np.sqrt(np.mean((oof_u.values - y_true.values) ** 2)))
y_val_true = ys_input['validation'].set_index(KEY_COL)[TARGET_COL]
val_u = final['val_pred_unit'].set_index(KEY_COL)['pred'].loc[y_val_true.index]
val_rmse = float(np.sqrt(np.mean((val_u.values - y_val_true.values) ** 2)))
y_test_true = ys_input['test'].set_index(KEY_COL)[TARGET_COL]
test_u = final['test_pred_unit'].set_index(KEY_COL)['pred'].loc[y_test_true.index]
test_rmse = float(np.sqrt(np.mean((test_u.values - y_test_true.values) ** 2)))
print('[Refit 완료] (original space, postprocess 이전)')
print(f'  OOF  unit RMSE = {oof_rmse:.6f}')
print(f'  val  unit RMSE = {val_rmse:.6f}')
print(f'  test unit RMSE = {test_rmse:.6f}')

## 5. 후처리 + 산출물 저장

In [ ]:
# fold_models.pkl + best_params.json + die/unit CSV 6개 저장. postprocess_config 주면 unit CSV는 튜닝값.
hpo.save_artifacts(
    refit_result=final,
    xs_train=xs_train, xs_val=xs_val, xs_test=xs_test,
    out_dir=OUT_DIR, exp_id=EXP_ID,
    feature_names=feat_cols_clean,
    extra_feature_name=None,
    y_train_unit=ys_input['train'],
    y_val_unit=ys_input['validation'],
    y_test_unit=ys_input['test'],
    postprocess_config=POSTPROCESS_CONFIG,
    study_meta=study_meta_for_save,
)
for f in sorted(os.listdir(OUT_DIR)):
    size_kb = os.path.getsize(os.path.join(OUT_DIR, f)) / 1024
    print(f'  {f:34s}  {size_kb:10,.1f} KB')